In [26]:
import chess
import torch
import torch.nn as nn
import torch.nn.functional as f
import pygame

if torch.cuda.is_available():
	print("PyTorch is using the GPU")
	GPUCount = torch.cuda.device_count()
	print(f"Found {GPUCount} GPUs")

	for i in range(GPUCount):
		print(f"GPU {i} found: {torch.cuda.get_device_name(i)}")

	device = torch.device("cuda:0")
else:
	print("PyTorch is using the CPU")
	device = torch.device("cpu")

print(f"Selected Device: {device}")

PyTorch is using the GPU
Found 1 GPUs
GPU 0 found: NVIDIA GeForce RTX 5070 Laptop GPU
Selected Device: cuda:0


In [27]:
pygame.init()
BoardSize = 512
SquareSize = BoardSize//8
DarkColor = (90,90,90)
LightColor = (205,205,205)
ChessBoard = chess.Board()
ChessBoardGUI = pygame.display.set_mode((BoardSize,BoardSize))
turn = 0
images = {}
ranks = [8,7,6,5,4,3,2,1]
files = ['a','b','c','d','e','f','g','h']

In [28]:
def LoadImages(SquareSize):
	pieces = {'K','Q','B','N','R','P','k','q','b','n','r','p'}

	for piece in pieces:
		img = pygame.image.load(f'ChessSprites/{piece}.png')
		img = pygame.transform.scale(img,(SquareSize,SquareSize))
		images[piece] = img
	return

In [29]:
def DrawBoard(ChessBoardGUI,SquareSize):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			Color = LightColor if ((row+col)%2==0) else DarkColor
			pygame.draw.rect(ChessBoardGUI,Color,(x,y,SquareSize,SquareSize))
	return

In [30]:
def DrawPieces(ChessBoardGUI,ChessBoard):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			SquareIdx = chess.square(col,7-row)
			piece = ChessBoard.piece_at(SquareIdx)
			if (piece is not None):
				ChessBoardGUI.blit(images[piece.symbol()],(x,y))

In [31]:
def ProcessChessData(FENString):
	tensor = torch.zeros((14,8,8), dtype=torch.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	return tensor

In [32]:
class ResidualBlock(nn.Module):
	def __init__(self,NumChannels):
		super().__init__()
		self.conv1 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn1 = nn.BatchNorm2d(NumChannels)

		self.conv2 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn2 = nn.BatchNorm2d(NumChannels)

	def forward(self,x):
		residual = x
		
		x = f.relu(self.bn1(self.conv1(x)))

		x = self.bn2(self.conv2(x))

		x += residual

		return f.relu(x)
	
class ChessNet(nn.Module):
	def __init__(self):
		super().__init__()

		self.ConvInput = nn.Conv2d(in_channels=14,out_channels=128,kernel_size=3,padding=1)
		self.BnInput = nn.BatchNorm2d(128)

		self.ResTower = nn.Sequential(
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128)
		)

		self.ConvValue = nn.Conv2d(in_channels=128,out_channels=1,kernel_size=1)
		self.BnValue = nn.BatchNorm2d(1)

		self.flat = nn.Flatten()

		self.fc1 = nn.Linear(64,128)
		self.fc2 = nn.Linear(128,1)

	def forward(self,x):
		x = f.relu(self.BnInput(self.ConvInput(x)))

		x = self.ResTower(x)

		x = f.relu(self.BnValue(self.ConvValue(x)))
		x = self.flat(x)
		x = f.relu(self.fc1(x))

		x = torch.tanh(self.fc2(x))

		return x
	
model = ChessNet()
model.to(device)
print(model)
model.load_state_dict(torch.load('ChessModel.pth'))
model.eval()

ChessNet(
  (ConvInput): Conv2d(14, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (BnInput): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (ResTower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): ResidualBlo

ChessNet(
  (ConvInput): Conv2d(14, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (BnInput): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (ResTower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): ResidualBlo

In [33]:
LoadImages(SquareSize)
PlayerClicks = []

try:
	with torch.no_grad():
		while not ChessBoard.is_game_over():
			for event in pygame.event.get():
				if event.type == pygame.QUIT:
					break
				elif event.type == pygame.MOUSEBUTTONDOWN and turn == 0:
					x,y = pygame.mouse.get_pos()
					col = x//SquareSize
					row = y//SquareSize
					ClickedSquare = files[col]+str(ranks[row])
					PlayerClicks.append(ClickedSquare)
					if len(PlayerClicks) == 2:
						if PlayerClicks[0] == PlayerClicks[1]:
							PlayerClicks = []
						else:
							move = PlayerClicks[0]+PlayerClicks[1]
							altmove = move+'q'
							move = chess.Move.from_uci(move)
							altmove = chess.Move.from_uci(altmove)
							if move in ChessBoard.legal_moves:
								ChessBoard.push(move)
								turn = 1
							elif altmove in ChessBoard.legal_moves:
								ChessBoard.push(altmove)
								turn = 1
							PlayerClicks = []
			if turn == 1:
				best = None
				score = float('inf')
				for move in ChessBoard.legal_moves:
					ChessBoard.push(move)
					InputFEN = ChessBoard.fen()
					ChessBoard.pop()
					InputTensor = ProcessChessData(InputFEN).to(device).unsqueeze(0)
					res = model(InputTensor).item()
					if (res < score):
						score = res
						best = move
				if best is not None:
					ChessBoard.push(best)
				turn = 0

			DrawBoard(ChessBoardGUI,SquareSize)
			DrawPieces(ChessBoardGUI,ChessBoard)
			pygame.display.flip()
finally:
	pygame.quit()